In [ ]:
import pandas as pd
import sys
import os
from sklearn import set_config
import numpy as np
from lifelines.statistics import logrank_test
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.model_selection import train_test_split
from typing import Tuple
from sklearn.model_selection import GroupKFold
set_config(display="text")  # displays text representation of estimators

from sksurv.util import Surv
sys.path.append(os.path.abspath("../../"))
from src.utils.ConvertTextToCsv import TextToCsv
from src.utils.Preprocessing import Preprocessor
from src.utils.cox_models import Cox_regression, p_values_Cox_regression

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
pp = Preprocessor()
cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01)
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_keep = pp.clean_columns_dataset(df_clinical_data)



In [ ]:
list_df = pp.total_type_len_type_cancer(df_clinical_keep)
df_clinical_keep["Tumor-Cancer"] = list_df
df_mRNA_transformed = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")
df_rsem_z_scores = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt")
clean_df = pp.eliminate_nan_genes(df_rsem_z_scores, "Hugo_Symbol")



In [ ]:

"""expression = df_ESR1_merged["expression"].values
lsit = sorted(expression)
plt.plot((lsit)) # plotting by columns
plt.show()
"""

In [ ]:

"""
ESR1 = df_ESR1_merged["expression"].values.reshape(-1,1)

scaler = StandardScaler()
z_score = scaler.fit_transform(ESR1)

plt.figure(figsize=(12,5))
plt.title("box plot of the z score")
plt.boxplot(z_score)
plt.show()

plt.figure(figsize=(12,5))
plt.hist(z_score, bins=30)
plt.title("Distribution of the z score")
plt.xlabel("z-score")
plt.ylabel("Frecuencia")
plt.show()
print(f"Min number ESR1: {ESR1.min()}")
print(f"Max number ESR1: {ESR1.max()}")
#plt.xlabel(f"Max number in ESR1:{ESR1.max()} ")"""

In [ ]:
def split_data_set(df_mRNA : pd.DataFrame, 
                       df_clinical : pd.DataFrame,
                       gene : str) -> Tuple:
    
        df_single_gene  = pp.gene_to_long(df_mRNA, gene)
        
        df_gene_merged = df_single_gene.merge(df_clinical, on="Sample ID", how="inner")
        
        df_gene_merged["Overall Survival (Months)"] = pd.to_numeric(df_gene_merged["Overall Survival (Months)"])
        
        status = df_gene_merged["Overall Survival Status"].astype(str).str.strip()
        
        df_gene_merged["event"] = status.str.contains("DECEASED", na=False) 
    
        df_gene_merged = df_gene_merged.dropna(subset=["Overall Survival (Months)"])
        
        df_gene_merged["expression"] = pd.to_numeric(df_gene_merged["expression"], errors="coerce")
        
        X = df_gene_merged[["expression"]]
        
        Y_surv = Surv.from_dataframe(
        event="event",
        time="Overall Survival (Months)",
        data=df_gene_merged
        )
        
        return X, Y_surv, df_gene_merged

In [ ]:
def split_data_60_months(df_mRNA: pd.DataFrame, 
                         df_clinical: pd.DataFrame,
                         gene: str) -> Tuple:
    
    df_single_gene = pp.gene_to_long(df_mRNA, gene)
    
    df_gene_merged = df_single_gene.merge(df_clinical, on="Sample ID", how="inner")
    
    df_gene_merged["Overall Survival (Months)"] = pd.to_numeric(
        df_gene_merged["Overall Survival (Months)"], errors="coerce"
    )
    
    status = df_gene_merged["Overall Survival Status"].astype(str).str.strip()
    df_gene_merged["event"] = status.str.contains("DECEASED", na=False)
    
    df_gene_merged["event_60"] = df_gene_merged["event"].copy()
    df_gene_merged["time_60"] = np.minimum(df_gene_merged["Overall Survival (Months)"], 60)
    
    df_gene_merged.loc[
        df_gene_merged["Overall Survival (Months)"] > 60, "event_60"
    ] = False
    
    df_gene_merged["expression"] = pd.to_numeric(
        df_gene_merged["expression"], errors="coerce"
    )
    
    df_gene_merged = df_gene_merged.dropna(subset=["time_60", "expression"]).copy()
    
    X = df_gene_merged[["expression"]]
    
    Y_surv = Surv.from_dataframe(
        event="event_60",
        time="time_60",
        data=df_gene_merged
    )
    
    return X, Y_surv, df_gene_merged

<font size="4">Risk Curve, Survival Curve and P-value for ESR1</font>

In [ ]:
X_ESR1, Y_surv_ESR17, df_gene_merged_ESR1 = split_data_60_months(clean_df,df_clinical_keep, "ESR1")
X_train_ESR1, X_test_ESR1, Y_train_ESR1, Y_test_ESR1 = train_test_split(
    X_ESR1, Y_surv_ESR17, train_size=0.80, test_size=0.20, random_state=42
)
betas_MKI67, chp_predict_ESR1, survival_curve_ESR1, risk_curve_ESR1 = Cox_regression(X_train_ESR1,
                                                                                     Y_train_ESR1,
                                                                                     X_test_ESR1,
                                                                                     "ESR1",
                                                                                     draw_plot=True)

In [ ]:
df_life_ESR1 = df_gene_merged_ESR1[["expression", "event", "Overall Survival (Months)"]].copy()
df_life_ESR1["expression"] = pd.to_numeric(df_life_ESR1["expression"], errors="coerce")
df_life_ESR1["Overall Survival (Months)"] = pd.to_numeric(df_life_ESR1["Overall Survival (Months)"], errors="coerce")
df_life_ESR1 = df_life_ESR1.dropna(subset=["expression", "event", "Overall Survival (Months)"])
df_life_ESR1["time_60"] = np.minimum(df_life_ESR1["Overall Survival (Months)"], 60)
df_life_ESR1["event_60"] = df_life_ESR1["event"].copy()
df_life_ESR1.loc[df_life_ESR1["Overall Survival (Months)"] > 60, "event_60"] = False
df_life_ESR1_cox = df_life_ESR1[["expression", "time_60", "event_60"]].copy()
p_value_ESR1 = p_values_Cox_regression(
    df_life_ESR1_cox,
    event_col="event_60",
    duration_col="time_60"
)

p_value_ESR1

<font size="4">LogRank for ESR1</font>

In [ ]:
thr = df_gene_merged_ESR1["expression"].median()
low_group = df_gene_merged_ESR1[df_gene_merged_ESR1["expression"] < thr]
high_group = df_gene_merged_ESR1[df_gene_merged_ESR1["expression"] >= thr]

results = logrank_test(
        durations_A=low_group["time_60"],
        durations_B=high_group["time_60"],
        event_observed_A=low_group["event_60"],
        event_observed_B=high_group["event_60"]
    )
print(results)

<font size="4">Risk Curve, Survival Curve and P-value for AURKA</font>

In [ ]:
X_AURKA, Y_surv_AURKA, df_gene_merged_AURKA = split_data_60_months(clean_df,df_clinical_keep, "AURKA")
X_train_AURKA, X_test_AURKA, Y_train_AURKA, Y_test_ESR1 = train_test_split(
    X_AURKA, Y_surv_AURKA, train_size=0.80, test_size=0.20, random_state=42
)
betas_AURKA, chp_predict_AURKA, survival_curve_AURKA, risk_curve_AURKA = Cox_regression(X_train_AURKA, Y_train_AURKA, X_test_AURKA, "AURKA", draw_plot=True)

In [ ]:
df_life_AURKA = df_gene_merged_AURKA[["expression", "event", "Overall Survival (Months)"]].copy()
df_life_AURKA["expression"] = pd.to_numeric(df_life_AURKA["expression"], errors="coerce")
df_life_AURKA["Overall Survival (Months)"] = pd.to_numeric(df_life_AURKA["Overall Survival (Months)"], errors="coerce")
df_life_AURKA = df_life_AURKA.dropna(subset=["expression", "event", "Overall Survival (Months)"])
df_life_AURKA["time_60"] = np.minimum(df_life_AURKA["Overall Survival (Months)"], 60)
df_life_AURKA["event_60"] = df_life_AURKA["event"].copy()
df_life_AURKA.loc[df_life_AURKA["Overall Survival (Months)"] > 60, "event_60"] = False
df_life_AURKA_cox = df_life_AURKA[["expression", "time_60", "event_60"]].copy()
p_value_AURKA = p_values_Cox_regression(
    df_life_AURKA_cox,
    event_col="event_60",
    duration_col="time_60"
)

p_value_AURKA

In [ ]:
thr = df_gene_merged_AURKA["expression"].median()
low_group = df_gene_merged_AURKA[df_gene_merged_AURKA["expression"] < thr]
high_group = df_gene_merged_AURKA[df_gene_merged_AURKA["expression"] >= thr]

results = logrank_test(
        durations_A=low_group["time_60"],
        durations_B=high_group["time_60"],
        event_observed_A=low_group["event_60"],
        event_observed_B=high_group["event_60"]
    )
print(results)

<font size="4">Risk Curve, Survival Curve and P-value for MKI67</font>

In [ ]:
X_MKI67, Y_surv_MKI67, df_gene_merged_MKI67 = split_data_60_months(clean_df,df_clinical_keep, "MKI67")
X_train_MKI67, X_test_MKI67, Y_train_MKI67, Y_test_MKI67 = train_test_split(
    X_MKI67, Y_surv_MKI67, train_size=0.80, test_size=0.20, random_state=42
)
betas_MKI67, chp_predict_MKI67, survival_curve_MKI67, risk_curve_MKI67 = Cox_regression(X_train_MKI67, Y_train_MKI67, X_test_MKI67, "MKI67", draw_plot=True)

In [ ]:
df_life_MKI67 = df_gene_merged_MKI67[["expression", "event", "Overall Survival (Months)"]].copy()
df_life_MKI67["expression"] = pd.to_numeric(df_life_MKI67["expression"], errors="coerce")
df_life_MKI67["Overall Survival (Months)"] = pd.to_numeric(df_life_MKI67["Overall Survival (Months)"], errors="coerce")
df_life_MKI67 = df_life_MKI67.dropna(subset=["expression", "event", "Overall Survival (Months)"])
df_life_MKI67["time_60"] = np.minimum(df_life_MKI67["Overall Survival (Months)"], 60)
df_life_MKI67["event_60"] = df_life_MKI67["event"].copy()
df_life_MKI67.loc[df_life_MKI67["Overall Survival (Months)"] > 60, "event_60"] = False
df_life_MKI67_cox = df_life_MKI67[["expression", "time_60", "event_60"]].copy()
p_value_MKI67 = p_values_Cox_regression(
    df_life_MKI67_cox,
    event_col="event_60",
    duration_col="time_60"
)

p_value_MKI67

In [ ]:
thr = df_gene_merged_MKI67["expression"].median()
low_group = df_gene_merged_MKI67[df_gene_merged_MKI67["expression"] < thr]
high_group = df_gene_merged_MKI67[df_gene_merged_MKI67["expression"] >= thr]

results = logrank_test(
        durations_A=low_group["time_60"],
        durations_B=high_group["time_60"],
        event_observed_A=low_group["event_60"],
        event_observed_B=high_group["event_60"]
    )
print(results)

<font size="4">GroupKFold for all the genes</font>

In [ ]:
#GroupKFold
group_k_fold = GroupKFold(n_splits=10)
group_k_fold.get_n_splits()
df_merged = pp.merge_datasets(df_clinical_keep, df_mRNA_transformed)
expressions_genes_cols = df_merged.iloc[1:20441].sample(5, axis="columns")



In [ ]:
cols = ["Sample ID","Tumor-Cancer", "Overall Survival Status", "Overall Survival (Months)"] + list(expressions_genes_cols)

comparation_df = df_merged.loc[
    df_merged["Tumor-Cancer"].isin(["Luminal A", "Luminal B", "TNBC", "HER2-enriched"]),
    cols
]

comparation_df = pp.eliminate_zero_genes(comparation_df, "Tumor-Cancer", threshold=0.8)


In [ ]:
status = comparation_df["Overall Survival Status"].astype(str).str.strip()

comparation_df["event"] = status.str.contains("DECEASED", na=False)

comparation_df = comparation_df.dropna(subset=["Overall Survival (Months)"])

groups = comparation_df["Sample ID"]
comparation_df = comparation_df.drop(columns="Sample ID")

X = comparation_df.iloc[:, 3:-1]

Y = Surv.from_dataframe(
    event="event",
    time="Overall Survival (Months)",
    data=comparation_df
)

for i, (train_index, test_index) in enumerate(group_k_fold.split(X, Y, groups=groups)):
    print(f"Fold {i}:")
    X_train = X.iloc[train_index]
    Y_train = Y[train_index]
    
    X_test = X.iloc[test_index]
    betas, chp_predict, survival_curve, risk_curve = Cox_regression(X_train,
                                                                    Y_train,
                                                                    X_test,
                                                                    draw_plot=False
                                                                    ) 
    
    
    break

In [ ]:
print(betas)
print(chp_predict)
print(survival_curve)
print(risk_curve)

In [ ]:
genes_expression = [
    "TBC1D9",
    "SUSD3",
    "SLC39A6",
    "GFRA1",
    "SOX11",
    "GATA3",
    "SLC15A2",
    "NANOS1",
    "ZNF552",
    "ESR1",
    "NAT1",
    "NME3",
    "DNALI1",
    "AGR3",
    "CA12",
    "BCL2",
    "MKI67",
    "TP53",
    "ESR1",
    "AURKA"
]
genes_significant_p_value = []
genes_non_significant_p_value = []

for gene_sample in genes_expression:
    X, Y_surv, df_gene_merged = split_data_set(clean_df,df_clinical_keep, gene_sample)

    df_life = df_gene_merged[["expression", "event", "Overall Survival (Months)"]].copy()
    
    df_life["expression"] = pd.to_numeric(df_life["expression"], errors="coerce")

    df_life["Overall Survival (Months)"] = pd.to_numeric(df_life["Overall Survival (Months)"], errors="coerce").astype(float)
    
    df_life = df_life.dropna(subset=["expression", "event", "Overall Survival (Months)"])
    
    df_life["time_60"] = np.minimum(df_life["Overall Survival (Months)"], 60)
    
    df_life["event_60"] = df_life["event"].copy()
    
    
    df_life.loc[df_life["Overall Survival (Months)"] > 60,  "event_60"] = False
    
    df_cox_60_months = df_life[["expression", "time_60", "event_60"]]
    
    value = p_values_Cox_regression(df_cox_60_months,event_col="event_60", duration_col="time_60")
    
    p = value["p"].item()
    if p <= 0.05:
        genes_significant_p_value.append(f"Gene: {gene_sample} and the P-value: {p}")
    else:
        genes_non_significant_p_value.append(f"Gene: {gene_sample} and the P-value: {p}")

In [ ]:
genes_non_significant_p_value

In [ ]:
genes_significant_p_value

In [ ]:
from lifelines.statistics import logrank_test

results_list = []

for gene_sample in genes_expression:
    df_gene = pp.gene_to_long(clean_df, gene_sample)
    df_gene_merged = df_gene.merge(df_clinical_keep, on="Sample ID", how="inner")

    df_gene_merged["expression"] = pd.to_numeric(df_gene_merged["expression"], errors="coerce")
    df_gene_merged["Overall Survival (Months)"] = pd.to_numeric(
        df_gene_merged["Overall Survival (Months)"], errors="coerce"
    )

    status = df_gene_merged["Overall Survival Status"].astype(str).str.strip()
    df_gene_merged["event"] = status.str.contains("DECEASED", na=False)

    df_gene_merged = df_gene_merged.dropna(
        subset=["expression", "Overall Survival (Months)", "event"]
    ).copy()

    thr = df_gene_merged["expression"].median()

    low_group = df_gene_merged[df_gene_merged["expression"] < thr]
    high_group = df_gene_merged[df_gene_merged["expression"] >= thr]

    results = logrank_test(
        durations_A=low_group["Overall Survival (Months)"],
        durations_B=high_group["Overall Survival (Months)"],
        event_observed_A=low_group["event"],
        event_observed_B=high_group["event"]
    )

    results_list.append({
        "gene": gene_sample,
        "p_value": results.p_value,
        "significant": results.p_value <= 0.05
    })

df_results = pd.DataFrame(results_list)
print(df_results)